# Elimination growth with small multipliers
This integer family isolates growth during elimination from the sensitivity of a linear system. Change the dimension and compare the largest numbers produced with a direct condition estimate. The displayed elimination keeps the first row when pivot sizes tie.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

dimension = 12
assert 3 <= dimension <= 40

def example(n):
    A = np.eye(n)-np.tril(np.ones((n,n)),-1)
    A[:,-1] = 1.
    return A

def eliminate(A):
    U=A.copy()
    growth=[np.max(abs(U))]
    multipliers=[]
    for j in range(len(U)-1):
        k=j+np.argmax(abs(U[j:,j]))
        U[[j,k]]=U[[k,j]]
        m=U[j+1:,j]/U[j,j]
        multipliers.extend(m)
        U[j+1:,j:] -= m[:,None]*U[j,j:]
        growth.append(np.max(abs(U)))
    return U,np.array(growth),max(abs(np.array(multipliers)))

A=example(dimension)
U,history,mmax=eliminate(A)
assert np.allclose(history,2.**np.arange(dimension))
print("Growth factor:",history[-1]," largest multiplier:",mmax)
print("2-norm condition number:",np.linalg.cond(A))

## Compare the pivot bound with the growth and conditioning
The exponential growth is exact in binary for these small integer inputs. It gives a possible roundoff hazard for perturbed problems; it does not prove large forward error in this exact demonstration.

In [ ]:
fig,(steps,scale)=plt.subplots(1,2,figsize=(10,4),layout="constrained")
steps.semilogy(range(dimension),history,"o-",color="#145c71")
steps.set(xlabel="Eliminated columns",ylabel="Largest absolute entry",title="The last column repeatedly doubles")
nn=np.arange(3,dimension+1)
scale.semilogy(nn,2.**(nn-1),"o-",color="#145c71",label="growth factor")
scale.semilogy(nn,[np.linalg.cond(example(i)) for i in nn],"s--",color="#b94626",label="condition number")
scale.set(xlabel="Matrix dimension",ylabel="Dimensionless factor",title="Different questions about a system")
scale.legend()
for ax in (steps,scale):ax.grid(alpha=.2)
plt.show()

## Challenge
Reorder the rows before calling eliminate. What changes in the growth history when pivot ties are resolved differently? Which singular values remain unchanged under the row permutation?

In [ ]:
_,reverse,_=eliminate(A[::-1])
assert np.isclose(np.linalg.cond(A),np.linalg.cond(A[::-1]))
print("After reversing the rows, observed growth becomes",reverse[-1])